# Logit Lens and Tuned Lens

**Author:** Raphaël Bernas

Lens methods show how a model's prediction develops across transformer blocks. `AllLayersSplitter` captures every residual-stream state in one trace, and the lens projects all of them together.

In [1]:
import warnings

warnings.filterwarnings("ignore", message="IProgress not found.*")

In [2]:
from transformers import AutoModelForSequenceClassification
from transformers.utils import logging as transformers_logging

from interpreto import AllLayersSplitter, LogitLens, TunedLens, plot_lens

transformers_logging.set_verbosity_error()
transformers_logging.disable_progress_bar()

## Classification with Logit Lens

Logit Lens can expose how class rankings develop through a sequence-classification model. Its `activation_names` list describes the order of the returned results.

In [3]:
classification_splitter = AllLayersSplitter(
    "distilbert-base-uncased-finetuned-sst-2-english",
    automodel=AutoModelForSequenceClassification,
)
classification_text = "The explanations are clear and useful."

logit_lens = LogitLens(classification_splitter, top_k=2)
logit_results = logit_lens(classification_text)
list(logit_results)

['model.distilbert.transformer.layer.0.input',
 'model.distilbert.transformer.layer.0',
 'model.distilbert.transformer.layer.1',
 'model.distilbert.transformer.layer.2',
 'model.distilbert.transformer.layer.3',
 'model.distilbert.transformer.layer.4',
 'model.distilbert.transformer.layer.5']

In [4]:
plot_lens(
    logit_results,
    classification_text,
    tokenizer=classification_splitter.tokenizer,
    label_names={0: "negative", 1: "positive"},
)

Class,Score
positive,0.511
negative,0.489
Class,Score
positive,0.517
negative,0.483
Class,Score
negative,0.535
positive,0.465
Class,Score
negative,0.571


## Generation with Tuned Lens

A Tuned Lens learns one residual affine translator for each non-final state. The translators are trained jointly against the language model's final prediction distribution. Training texts are processed sequentially, while all depths from one text share a prediction-head call.

In [5]:
generation_splitter = AllLayersSplitter("distilgpt2")
training_texts = [
    "Paris is the capital of France.",
    "Rome is the capital of Italy.",
]

tuned_lens = TunedLens(generation_splitter, top_k=3)
losses = tuned_lens.fit(training_texts, epochs=1)
losses

[7.739313364028931]

In [6]:
generation_text = "Paris is the capital of"
tuned_results = tuned_lens(generation_text)
plot_lens(tuned_results, generation_text, tokenizer=generation_splitter.tokenizer)

Input token,Top predictions
Paris,"ebook (0.186), ignty (0.155), <|endoftext|> (0.116)"
is,"� (0.933), is (0.0202), challeng (0.00544)"
the,"the (0.875), its (0.0912), a (0.00941)"
capital,"capital (0.99), capitals (0.00486), of (0.00354)"
of,"of (0.979), luster (0.0178), etheless (0.000815)"
Input token,Top predictions
Paris,". (0.0477), The (0.0276), and (0.0264)"
is,"a (0.406), in (0.0753), an (0.0624)"
the,"world (0.0735), first (0.0413), winner (0.0335)"
capital,"of (1), on (3.57e-07), upon (2.51e-07)"


`TunedLens` is a regular PyTorch module. Save and restore its translators with `state_dict()` and `load_state_dict()`.